In [590]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [591]:
np.random.seed(0)

In [592]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [593]:
def best_response(x, thresholds, priors, c):
    posteriors = bayesian_update(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

In [594]:
def best_response_vectorized(X, thresholds, priors, c):
    posteriors = bayesian_update(priors)

    thresholds = np.array(thresholds)
    X = np.array(X)

    utilities_cumsum = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_cumsum[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [595]:
def merge_classifiers(X, X_p, thresholds, priors):
    merged_thresholds, merged_priors = deepcopy(thresholds), deepcopy(priors)
    for _ in range(len(thresholds)):
        support = np.array([np.mean(X_p==threshold) for threshold in merged_thresholds])
        dominated = np.where(support == 0)[0]

        if len(dominated) == 0:
            break
        
        for i in reversed(dominated):
            threshold_i = merged_thresholds[i]

            jumps = X_p[X < threshold_i]
            if len(jumps) == 0:
                continue

            candidates = jumps[jumps > threshold_i]
            if len(candidates) == 0:
                continue

            values, counts = np.unique(candidates, return_counts=True)
            threshold_dom = values[np.argmax(counts)]

            dom_idx = np.where(threshold_dom == merged_thresholds)[0][0]

            merged_priors[dom_idx] += merged_priors[i]
            merged_priors = np.delete(merged_priors, i)
            merged_thresholds = np.delete(merged_thresholds, i)

    return merged_thresholds, merged_priors

In [596]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [597]:
def accuracy_loss(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    losses = np.empty_like(thresholds)
    for i, threshold in enumerate(thresholds):
        Y_p = (X_p >= threshold).astype(float)
        losses[i] = np.abs(Y_true - Y_p).mean()
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [598]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    # X_p = np.array([best_response(x, thresholds_p, priors_p, c) for x in X])
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    thresholds_p, priors_p = merge_classifiers(X, X_p, thresholds_p, priors_p)
    acc_loss_p = accuracy_loss(X, X_p, thresholds_p, priors_p, threshold_true)
    return acc_loss_p

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [599]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [600]:
def find_partitions_greedy(X, thresholds, priors, threshold_true, c):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1


    Q = collections.deque(itertools.combinations(P.keys(), 2))

    while Q:
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for c_id in P.keys():
                if c_id != new_id:
                    Q.append((new_id, c_id))
    return list(P.values())

In [ ]:
# def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
#     indices = [i for i in range(len(thresholds))]
#     partitions_set = list(set_partitions(indices))

#     best_partition = None
#     best_loss = np.inf

#     for partitions in tqdm.tqdm(partitions_set):
#         acc_loss = 0.
#         for partition in partitions:
#             acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
#             acc_loss += acc_loss_p * np.sum(priors[partition])

#         if acc_loss < best_loss:
#             best_loss = acc_loss
#             best_partition = deepcopy(partitions)

#     return best_partition

In [602]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            X, partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in tqdm.tqdm(partitions_set):
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [603]:
c = 5.0
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)
X = np.arange(threshold_min, threshold_max, 1e-6).round(4)

# priors = np.zeros_like(thresholds)
priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])
# balance_priors(priors, random=True)

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.100,0.200,0.300,0.40,0.500,0.600,0.700,0.800,0.90
priors,0.114,0.115,0.036,0.25,0.022,0.054,0.062,0.097,0.25


In [604]:
partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
# partition_greedy = [[0,1,2,3,4,5],[6],[7],[8]]
partition_optimal = find_partitions_optimal(X, thresholds, priors, threshold_true, c)

acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
acc_loss_optimal = evaluate_system(X, partition_optimal, thresholds, priors, threshold_true, c)

100%|██████████| 21147/21147 [00:39<00:00, 538.56it/s]


In [608]:
print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy
------
Partition: [[0, 1, 2, 3, 4, 5, 6, 8], [7]]
Acc Loss : 0.2444

Optimal
-------
Partition: [[0, 1, 2, 3, 4], [5, 6], [7], [8]]
Acc Loss : 0.2056


In [607]:
p = [[0,1,2,3,4,5],[6],[7],[8]]
evaluate_system(X, p, thresholds, priors, threshold_true, c)

np.float64(0.20691203406774)